In [44]:
# Cell: Load and inspect a sample run parquet
import sys
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Add project root to sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Import environment utilities
from src.environment import compute_nash_and_monopoly_static, get_demand_and_profit_static

# Setup Plotting Style
sns.set_context("talk")
plt.rcParams['figure.figsize'] = (20, 16)

In [45]:
def find_experiment_dir(strats, mu, k, N, run):
    """
    Find the experiment directory matching the given parameters.
    
    Parameters:
    -----------
    strats : int
        Number of strategies (3 or 4)
    mu : float
        Learning rate parameter
    k : int
        K parameter
    N : int
        Number of sellers
    run : int
        Run number
    
    Returns:
    --------
    Path or None
        The experiment directory path if found, None otherwise
    """
    results_dir = project_root / "data" / "results"
    
    # Construct the expected directory name pattern
    strat_str = f"{strats}strats"
    mu_str = f"mu{mu}"
    k_str = f"k{k}"
    N_str = f"N_{N}"
    run_str = f"run_{run}.parquet"
    # Search for matching directory
    for exp_dir in results_dir.iterdir():
        if not exp_dir.is_dir():
            continue
        
        dir_name = exp_dir.name
        if strat_str in dir_name and mu_str in dir_name and k_str in dir_name:
            return exp_dir / N_str / run_str
    
    return None


In [46]:
strats = 4      # 3 or 4
N = 10           # Number of sellers
k = 30          # K parameter
mu = 0.22       # competitiveness
run = 14          # Run number
parquet_dir = find_experiment_dir(strats, mu, k, N, run)

df = pd.read_parquet(parquet_dir)

In [47]:
display(df.head())
print("Columns:", df.columns.tolist())

,run_id,t_global,episode,step_in_k,price_min,price_mean,delta,converged,is_cycle,a_0,...,pi_6,a_7,p_7,pi_7,a_8,p_8,pi_8,a_9,p_9,pi_9
0,14,0,507899,0,2.315937,2.315937,0.950528,True,False,0,...,0.092644,0,2.315937,0.092644,0,2.315937,0.092644,0,2.315937,0.092644
1,14,1,507899,1,2.181987,2.181987,1.000000,True,False,0,...,0.096199,0,2.181987,0.096199,0,2.181987,0.096199,0,2.181987,0.096199
2,14,2,507899,2,2.048037,2.048037,0.958377,True,False,0,...,0.093208,0,2.048037,0.093208,0,2.048037,0.093208,0,2.048037,0.093208
3,14,3,507899,3,1.914086,1.914086,0.852685,True,False,0,...,0.085615,0,1.914086,0.085615,0,1.914086,0.085615,0,1.914086,0.085615
4,14,4,507899,4,1.780136,1.780136,0.708327,True,False,0,...,0.075244,0,1.780136,0.075244,0,1.780136,0.075244,0,1.780136,0.075244


Columns: ['run_id', 't_global', 'episode', 'step_in_k', 'price_min', 'price_mean', 'delta', 'converged', 'is_cycle', 'a_0', 'p_0', 'pi_0', 'a_1', 'p_1', 'pi_1', 'a_2', 'p_2', 'pi_2', 'a_3', 'p_3', 'pi_3', 'a_4', 'p_4', 'pi_4', 'a_5', 'p_5', 'pi_5', 'a_6', 'p_6', 'pi_6', 'a_7', 'p_7', 'pi_7', 'a_8', 'p_8', 'pi_8', 'a_9', 'p_9', 'pi_9']


In [48]:
def find_config_file(strats, mu, k, N):
    """
    Find the config JSON file for given parameters.
    
    Parameters:
    -----------
    strats : int
        Number of strategies (3 or 4)
    mu : float
        Learning rate parameter
    k : int
        K parameter
    N : int
        Number of sellers
    
    Returns:
    --------
    Path or None
        The config file path if found, None otherwise
    """
    results_dir = project_root / "data" / "results"
    
    # Construct the expected directory name pattern
    strat_str = f"{strats}strats"
    mu_str = f"mu{mu}"
    k_str = f"k{k}"
    
    # Search for matching experiment directory
    for exp_dir in results_dir.iterdir():
        if not exp_dir.is_dir():
            continue
        
        dir_name = exp_dir.name
        if strat_str in dir_name and mu_str in dir_name and k_str in dir_name:
            # Found the experiment directory
            config_file = exp_dir / f"Config_N_{N}.json"
            if config_file.exists():
                return config_file
    
    return None


def load_and_compute_grid(strats, mu, k, N):
    """
    Load config file and compute the price grid for given parameters.
    
    Parameters:
    -----------
    strats : int
        Number of strategies (3 or 4)
    mu : float
        Learning rate parameter
    k : int
        K parameter
    N : int
        Number of sellers
    
    Returns:
    --------
    dict
        Dictionary containing:
        - 'config': The loaded config dictionary
        - 'K': K parameter
        - 'price_grid': Array of 10 grid prices
    """
    # Find config file
    config_file = find_config_file(strats, mu, k, N)
    if config_file is None:
        return None
    
    # Load config
    with open(config_file, 'r') as f:
        cfg = json.load(f)
    
    # Extract parameters
    num_sellers = cfg['num_sellers']
    a_val = cfg['a_val']
    c_val = cfg['c_val']
    mu_val = cfg['mu']
    a0 = cfg['a0']
    
    # Compute Nash and Monopoly prices
    p_nash, p_monopoly = compute_nash_and_monopoly_static(
        num_sellers=num_sellers,
        a_val=a_val,
        mu=mu_val,
        a0=a0,
        c_val=c_val
    )
    
    # Build price grid (10 grids, Nash at index 1, Monopoly at index 8)
    step = (p_monopoly - p_nash) / 7  # (10 - 3) = 7
    price_grid = np.linspace(p_nash - step, p_monopoly + step, 10)
    
    return {
        'config': cfg,
        'K': cfg['K'],
        'price_grid': price_grid
    }


def classify_equilibrium(df, grid_info):
    """
    Classify equilibrium based on the states visited.
    
    Classification rules:
    - 'H' (High): All states are in grids 7, 8, 9
    - 'L' (Low): All states are in grids 0, 1, 2
    - 'C' (Cycle): States include both High and Low, but no Mid (3-6)
    - 'O' (Other): Everything else
    
    Parameters:
    -----------
    df : pd.DataFrame
        The run data from parquet file
    grid_info : dict
        Grid information from load_and_compute_grid()
    
    Returns:
    --------
    str
        'H', 'L', 'C', or 'O'
    """
    K = grid_info['K']
    price_grid = grid_info['price_grid']
    
    # Get price columns
    p_cols = sorted([c for c in df.columns if c.startswith('p_')],
                    key=lambda x: int(x.split('_')[1]))
    
    # Extract rows at end of each K-period (step_in_k == K-1)
    end_rows = df[df['step_in_k'] == K - 1]
    
    # Collect all state grid indices
    state_grids = []
    for _, row in end_rows.iterrows():
        prices = [row[col] for col in p_cols]
        min_price = min(prices)
        # Find closest grid point
        grid_idx = np.argmin(np.abs(price_grid - min_price))
        state_grids.append(grid_idx)
    
    # Categorize each state
    has_high = any(g in [7, 8, 9] for g in state_grids)
    has_mid = any(g in [3, 4, 5, 6] for g in state_grids)
    has_low = any(g in [0, 1, 2] for g in state_grids)
    
    # Classify
    if has_high and not has_mid and not has_low:
        return 'H'
    elif has_low and not has_mid and not has_high:
        return 'L'
    elif (has_high and has_low) and not has_mid:
        return 'C'
    else:
        return 'O'

In [49]:
# Load grid info
grid_info = load_and_compute_grid(strats=strats, mu=mu, k=k, N=N)

# Classify the equilibrium
eq_class = classify_equilibrium(df, grid_info)

print(f"Equilibrium Type: {eq_class}")
print()
print("Classification:")
print("  H = High (Collusive) - all states in grids 7, 8, 9")
print("  L = Low (Competitive) - all states in grids 0, 1, 2")
print("  C = Cycle - states in both High and Low, but no Mid (3-6)")
print("  O = Other - everything else")

Equilibrium Type: C

Classification:
  H = High (Collusive) - all states in grids 7, 8, 9
  L = Low (Competitive) - all states in grids 0, 1, 2
  C = Cycle - states in both High and Low, but no Mid (3-6)
  O = Other - everything else


## Equilibrium Classification

Simple classification based on state grid levels.